In [4]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from prophet import Prophet
from sklearn.metrics import mean_squared_error, mean_absolute_error

input_dir = "../Data/Prophet_Preprocessed"
output_dir = "Prophet_Results"
os.makedirs(output_dir, exist_ok=True)

summary = []

for file in os.listdir(input_dir):
    if not file.endswith("_prophet.csv"):
        continue
    stock = file.replace("_prophet.csv", "")
    print(f"\n🔄 Processing {stock}...")

    df = pd.read_csv(os.path.join(input_dir, file))
    df['ds'] = pd.to_datetime(df['ds'])
    df = df.dropna(subset=['y'])

    # Optional: log-transform for high variance
    use_log = False
    if df['y'].min() > 0 and df['y'].max() / df['y'].min() > 10:
        df['y'] = np.log(df['y'])
        use_log = True

    # Train-test split (80/20)
    split_idx = int(len(df) * 0.8)
    train_df = df.iloc[:split_idx]
    test_df = df.iloc[split_idx:]

    # Prophet model with tuned settings
    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
        seasonality_mode='multiplicative',
        changepoint_prior_scale=0.1
    )
    model.fit(train_df)

    # Create future dataframe for forecasting (same length as test)
    future = model.make_future_dataframe(periods=len(test_df), freq='B')
    forecast = model.predict(future)

    # Extract forecast for test period
    forecast_test = forecast.iloc[-len(test_df):][['ds', 'yhat']].set_index('ds')
    test_actual = test_df.set_index('ds')['y']

    # Inverse log-transform if used
    if use_log:
        forecast_test['yhat'] = np.exp(forecast_test['yhat'])
        test_actual = np.exp(test_actual)
        train_df['y'] = np.exp(train_df['y'])
        test_df['y'] = np.exp(test_df['y'])

    # Evaluation
    rmse = np.sqrt(mean_squared_error(test_actual, forecast_test['yhat']))
    mae = mean_absolute_error(test_actual, forecast_test['yhat'])
    print(f"Test RMSE: {rmse:.4f} | MAE: {mae:.4f}")
    summary.append({'Stock': stock, 'RMSE': rmse, 'MAE': mae})

    # Visualization (full train and test period)
    plt.figure(figsize=(12, 6))
    plt.plot(train_df['ds'], train_df['y'], label="Train", color="blue", linewidth=2)
    plt.plot(test_df['ds'], test_df['y'], label="Test Actual", color="green", linewidth=2)
    plt.plot(test_df['ds'], forecast_test['yhat'], label="Test Forecast", color="red", linestyle='--', linewidth=2)
    plt.title(f"{stock} Prophet Forecast")
    plt.xlabel("Date")
    plt.ylabel("Price")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{output_dir}/{stock}_prophet_forecast.png")
    plt.close()

    # Model diagnostics: plot components
    fig2 = model.plot_components(forecast)
    fig2.savefig(f"{output_dir}/{stock}_prophet_components.png")
    plt.close(fig2)

    # Save forecast results
    out_df = pd.DataFrame({
        "Date": test_df['ds'],
        "Actual": test_df['y'],
        "Forecast": forecast_test['yhat'].values
    })
    out_df.to_csv(f"{output_dir}/{stock}_prophet_forecast.csv", index=False)
    print(f"✅ Saved results for {stock}")

# Save RMSE/MAE summary for all stocks to CSV
summary_df = pd.DataFrame(summary)
summary_df.to_csv(f"{output_dir}/prophet_rmse_mae_summary.csv", index=False)
print("\n📊 RMSE/MAE summary for all stocks:")
print(summary_df)

print("\n✅ All stocks processed. Prophet results and diagnostics saved in Prophet_Results/")


🔄 Processing Apple...


19:56:52 - cmdstanpy - INFO - Chain [1] start processing
19:56:54 - cmdstanpy - INFO - Chain [1] done processing
C:\Users\HP\AppData\Local\Temp\ipykernel_1724\156637569.py:57: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df['y'] = np.exp(train_df['y'])
C:\Users\HP\AppData\Local\Temp\ipykernel_1724\156637569.py:58: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['y'] = np.exp(test_df['y'])


Test RMSE: 220.5416 | MAE: 173.1168
✅ Saved results for Apple

🔄 Processing GeneralElectric...


19:56:57 - cmdstanpy - INFO - Chain [1] start processing
19:57:02 - cmdstanpy - INFO - Chain [1] done processing
C:\Users\HP\AppData\Local\Temp\ipykernel_1724\156637569.py:57: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df['y'] = np.exp(train_df['y'])
C:\Users\HP\AppData\Local\Temp\ipykernel_1724\156637569.py:58: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['y'] = np.exp(test_df['y'])


Test RMSE: 87.9133 | MAE: 63.9492
✅ Saved results for GeneralElectric

🔄 Processing IBM...


...

Test RMSE: 71.6479 | MAE: 52.2737
✅ Saved results for IBM

🔄 Processing Johnson&Johnson...


19:57:12 - cmdstanpy - INFO - Chain [1] start processing
19:57:14 - cmdstanpy - INFO - Chain [1] done processing


Test RMSE: 22.8844 | MAE: 18.5075
✅ Saved results for Johnson&Johnson

🔄 Processing Microsoft...


19:57:17 - cmdstanpy - INFO - Chain [1] start processing
19:57:20 - cmdstanpy - INFO - Chain [1] done processing
C:\Users\HP\AppData\Local\Temp\ipykernel_1724\156637569.py:57: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df['y'] = np.exp(train_df['y'])
C:\Users\HP\AppData\Local\Temp\ipykernel_1724\156637569.py:58: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['y'] = np.exp(test_df['y'])


Test RMSE: 272.1464 | MAE: 223.9676
✅ Saved results for Microsoft

📊 RMSE/MAE summary for all stocks:
             Stock        RMSE         MAE
0            Apple  220.541571  173.116783
1  GeneralElectric   87.913347   63.949244
2              IBM   71.647932   52.273698
3  Johnson&Johnson   22.884402   18.507506
4        Microsoft  272.146429  223.967611

✅ All stocks processed. Prophet results and diagnostics saved in Prophet_Results/


In [6]:
# Save RMSE/MAE summary for all stocks to CSV
summary_df = pd.DataFrame(summary)
summary_df.to_csv(f"{output_dir}/prophet_rmse_mae_summary.csv", index=False)
print("\n📊 RMSE/MAE summary for all stocks saved")


📊 RMSE/MAE summary for all stocks saved
